In [38]:


from datetime import date
from typing import Dict, Self
from sqlmodel import SQLModel, Field
from sqlalchemy.types import String
from pydantic import HttpUrl
from pathlib import Path

# Assume MEDIUM_HOSTS and logger are defined elsewhere

class InputContent(SQLModel, table=True):
	id: int | None = Field(default=None, primary_key=True)  # Add an ID as primary key
	source: HttpUrl | Path = Field(unique=True, sa_type=String(2048))
	upload_date: date
	already_read: bool = Field(default=False)
	read_priority: int = Field(default=1, ge=0, le=5)
	provenance: str | None = Field(default=None)  # Use Optional for clarity

/Users/tdurouchoux/opt/anaconda3/envs/py311_dsview/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.InputContent, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


InvalidRequestError: Table 'inputcontent' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

In [3]:
from typing import List

In [4]:
tags = ["ML", "DL"]

In [36]:
content = InputContent(source="https://towardsdatascience.com/automated-detection-of-data-quality-issues-54a3cb283a91", upload_date=date.today())

In [37]:
content.source

'https://towardsdatascience.com/automated-detection-of-data-quality-issues-54a3cb283a91'

In [34]:
valid_content = InputContent.model_validate(content)

In [35]:
valid_content.source

'https://towardsdatascience.com/automated-detection-of-data-quality-issues-54a3cb283a91'

In [7]:
type(content.link)

pydantic_core._pydantic_core.Url

In [4]:
type(content.link)

str

In [13]:
from datetime import date
import logging
from pathlib import Path
from typing import Dict, Union

from pydantic import BaseModel, Field, HttpUrl

MEDIUM_HOSTS = ["medium.com", "towardsdatascience.com"]

logger = logging.getLogger(__name__)


class InputContent(BaseModel):
    link: Union[HttpUrl, Path]
    upload_date: date
    already_read: bool = False
    read_priority: int = Field(default=1, ge=0, le=5)
    source: str = None

    def model_post_init(self, __context):
        try:
            if self.link.host in MEDIUM_HOSTS:
                logger.info("Received a medium link, redirecting to readmedium")
                self.link = HttpUrl("https://readmedium.com/" + str(self.link))

        except AttributeError:
            return

    def get_str_dict(self) -> Dict:
        instance_dict = dict(self)
        instance_dict["link"] = str(self.link)
        instance_dict["upload_date"] = self.upload_date.isoformat()

        if self.source is None:
            del instance_dict["source"]

        return instance_dict


In [39]:
import pandas as pd

In [40]:
pdf_snapshot = pd.read_json("../snapshot_content.json")

In [43]:
pdf_snapshot.head(30)

,already_read,link,read_priority,upload_date,source
0,False,https://abseil.io/resources/swe-book,1,2023-01-19,NaN
1,False,https://inseefrlab.github.io/formation-mlops/s...,1,2023-10-05,NaN
2,False,https://codelabs.developers.google.com/tensorf...,1,2023-09-21,NaN
3,False,https://stanford-cs329s.github.io/,1,2022-09-15,NaN
4,True,https://github.com/anthropics/courses/tree/mas...,4,2024-09-29,Aucune
5,False,https://carl-mcbride-ellis.github.io/TOBoML/TO...,1,2024-07-10,NaN
6,True,https://academy.langchain.com/courses/intro-to...,1,2024-09-19,Aucune
7,False,https://snap-stanford.github.io/cs224w-notes/?...,1,2024-04-24,NaN
8,False,https://stanford-cs324.github.io/winter2022/,1,2023-01-19,NaN
9,False,https://cognitiveclass.ai/,1,2023-09-28,NaN


In [47]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

In [52]:
pdf_snapshot.sort_values("upload_date", ascending=False).head(50)

,already_read,link,read_priority,upload_date,source
36,True,https://github.com/openai/swarm/tree/main,2,2024-10-15,Alpha Signal
29,True,https://github.com/Cinnamon/kotaemon,4,2024-10-15,Alpha Signal
55,True,https://cookbook.openai.com/examples/evaluation/how_to_eval_abstractive_summarization,0,2024-10-13,Aucune
26,True,https://github.com/Sinaptik-AI/pandas-ai,0,2024-10-12,Alpha Signal
152,True,https://www.stateof.ai/,5,2024-10-12,Alpha Signal
58,True,https://bnm3k.github.io/blog/duckdb-jit-udfs-numba,4,2024-10-12,Aucune
43,True,https://github.com/wasiahmad/Awesome-LLM-Synthetic-Data,3,2024-10-12,Alpha Signal
112,True,https://www.fharrell.com/post/cluster/?utm_source=substack&utm_medium=email,3,2024-10-12,Data science weekly
119,True,https://www.deep-ml.com/about,4,2024-10-12,Alpha Signal
128,True,https://www.government-transformation.com/data/splink-transforming-data-linking-through-open-source-collaboration,5,2024-10-09,Data Science Weekly


In [ ]:
pdf_snapshot.sort_values(["read_priority", "upload_date"], ascending=False).head(50)

,already_read,link,read_priority,upload_date,source
152,True,https://www.stateof.ai/,5,2024-10-12,Alpha Signal
128,True,https://www.government-transformation.com/data/splink-transforming-data-linking-through-open-source-collaboration,5,2024-10-09,Data Science Weekly
181,True,https://learn.microsoft.com/en-us/ai/playbook/technology-guidance/generative-ai/working-with-llms/evaluation/list-of-eval-metrics,5,2024-09-29,Aucune
16,False,https://github.com/mlabonne/llm-course,5,2024-01-04,NaN
29,True,https://github.com/Cinnamon/kotaemon,4,2024-10-15,Alpha Signal
58,True,https://bnm3k.github.io/blog/duckdb-jit-udfs-numba,4,2024-10-12,Aucune
119,True,https://www.deep-ml.com/about,4,2024-10-12,Alpha Signal
20,True,https://github.com/microsoft/generative-ai-for-beginners?tab=readme-ov-file,4,2024-10-08,Alpha Signal
48,True,https://github.com/KruxAI/ragbuilder?tab=readme-ov-file#installation,4,2024-10-08,Alpha Signal
56,True,https://sarahconstantin.substack.com/p/the-great-data-integration-schlep?utm_campaign=Data_Elixir&utm_source=Data_Elixir_505,4,2024-10-08,Data Elixir
